---
title: "Exercise: Build a RAG Pipeline from Scratch"
format: html
---


# Day 3 Lab Exercise — Build a RAG Pipeline from Scratch
**Estimated time: 60 minutes**

## Scenario
You are building a **Data Engineering Q&A Assistant** for the SDAIA bootcamp.
Students can ask it questions and it will retrieve the most relevant content from the course knowledge base and generate a grounded answer.

Your job is to implement each stage of the pipeline.
Six tasks, each building on the previous one.

## Pipeline you will build
```
Documents → Chunking → ChromaDB Vector Index ─┐
                     → BM25 Keyword Index    ─┴─► RRF Fusion → Reranking → RAG Prompt → Evaluation
```

**Do not modify cells marked `# ── PROVIDED ──`.**
Fill in every `# TODO` comment. Run the test cell after each task to verify.

## Setup

In [ ]:
!pip install chromadb sentence-transformers rank-bm25 numpy

In [ ]:
# ── PROVIDED ── imports (do not modify)
import re
import numpy as np
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

print("✅ All libraries imported successfully.")

## Knowledge Base
These are the documents your RAG system will index. Do not modify them.

In [ ]:
# ── PROVIDED ── knowledge base (do not modify)
DOCUMENTS = [
    {"id": "d01", "text": "Apache Kafka stores messages in topics. Each topic is split into partitions — ordered, append-only logs. Every message inside a partition gets an offset, which is its permanent address. A consumer group shares partitions so that each partition is assigned to exactly one member at a time."},
    {"id": "d02", "text": "A tumbling window is a fixed-size, non-overlapping time bucket. Every event belongs to exactly one tumbling window. A sliding window overlaps — the same event can appear in multiple windows. A session window closes after a configurable gap of inactivity, so its size varies per user."},
    {"id": "d03", "text": "A watermark is a threshold that tells the stream processor how long to wait for late-arriving events before closing a window. Events arriving after the watermark has passed are dropped or sent to a dead-letter queue. A larger watermark increases completeness but raises end-to-end latency."},
    {"id": "d04", "text": "Exactly-once delivery requires coordination between the producer, broker, and consumer. At-most-once loses data on failure. At-least-once retries on failure but may produce duplicates. Exactly-once prevents both, at a cost of roughly 20 percent throughput overhead."},
    {"id": "d05", "text": "Delta Lake stores data in Parquet files and records every write operation in a transaction log called the delta_log. Each entry in the log is a JSON commit file. Readers replay the log from version zero to reconstruct the current table state, which is how Delta provides ACID guarantees without a traditional database engine."},
    {"id": "d06", "text": "MERGE INTO allows you to apply inserts, updates, and deletes atomically in a single operation. It is the correct way to implement CDC (Change Data Capture) into a Delta table. Delta executes MERGE by scanning only the affected Parquet files and writing new ones, never modifying existing files in place."},
    {"id": "d07", "text": "HNSW stands for Hierarchical Navigable Small World. It is a graph-based approximate nearest-neighbour index used inside ChromaDB, Pinecone, and Weaviate. The M parameter controls how many bidirectional links each node has. The ef parameter at query time controls how large the candidate list is during search, trading recall for latency."},
    {"id": "d08", "text": "Hybrid search runs vector semantic search and BM25 keyword search in parallel, then merges the two result lists. Vector search finds semantically similar documents even when exact words differ. BM25 finds documents that share exact terms with the query. Hybrid search outperforms either method alone on most benchmarks."},
    {"id": "d09", "text": "Reciprocal Rank Fusion scores a document as the sum of 1 divided by k plus its rank across all result lists, where k equals 60 by convention. This formula rewards documents that rank consistently high in both the vector list and the BM25 list, without requiring any weights to be tuned manually."},
    {"id": "d10", "text": "A cross-encoder takes a query and a document as a pair and scores them jointly. This is more accurate than comparing independent embeddings because the model can see the interaction between query tokens and document tokens. The standard two-stage pattern is: retrieve the top 50 candidates with a bi-encoder, then rerank them with a cross-encoder, then pass the top 3 to the LLM."},
    {"id": "d11", "text": "RAG evaluation measures four properties. Context precision is the fraction of retrieved chunks that are actually relevant. Context recall measures whether all needed chunks were retrieved. Answer faithfulness checks whether the generated answer stays within the retrieved context. Answer relevance checks whether the answer addresses the user question."},
    {"id": "d12", "text": "A data contract is a machine-enforceable agreement between a data producer and its consumers. It specifies the schema, field types, and SLA. Pydantic v2 enforces contracts in Python using strict type checking. Breaking changes such as removing a column or narrowing a type require a major version bump and a migration window for downstream consumers."},
]

print(f"✅ Knowledge base loaded — {len(DOCUMENTS)} documents ready.")

## Task 1 — Document Chunking (10 min)

Split each document into **overlapping sentence chunks**.

Rules:
- Each chunk contains `chunk_size` consecutive sentences (default = 2).
- Adjacent chunks share **1 sentence** (overlap), so context is not cut off at boundaries.
- Example with `chunk_size=2` and sentences `[A, B, C, D]`:
  - Chunk 0 → `"A B"`
  - Chunk 1 → `"B C"` ← shares B with chunk 0
  - Chunk 2 → `"C D"` ← shares C with chunk 1

> **Hint:** The step between chunk start positions is `chunk_size - 1`.
> Use `re.split(r'(?<=[.!?])\s+', text)` to split into sentences.

In [ ]:
def chunk_documents(docs, chunk_size=2):
    """
    Split each document into overlapping sentence chunks.

    Args:
        docs       : list of dicts with keys 'id' and 'text'
        chunk_size : number of sentences per chunk

    Returns:
        list of dicts with keys 'id', 'text', 'doc_id'
    """
    all_chunks = []

    for doc in docs:
        sentences = re.split(r'(?<=[.!?])\s+', doc["text"].strip())

        # TODO: iterate through sentences in steps of (chunk_size - 1)
        #       build a chunk string by joining chunk_size consecutive sentences
        #       append a dict with keys: id, text, doc_id
        #       id format: f"{doc['id']}_c{i:02d}"
        for i in range(0, ???, ???):
            chunk_text = ???
            if not chunk_text.strip():
                continue
            all_chunks.append({
                "id":     ???,
                "text":   ???,
                "doc_id": ???,
            })

    return all_chunks

In [ ]:
# ── TEST 1 ── run this to check your work
chunks = chunk_documents(DOCUMENTS, chunk_size=2)

assert len(chunks) > len(DOCUMENTS), "You should have more chunks than documents after splitting."
assert all("id" in c and "text" in c and "doc_id" in c for c in chunks), "Each chunk must have id, text, doc_id."
assert any("_c01" in c["id"] for c in chunks), "Chunk IDs should follow the format doc_id_c01, _c02, etc."

print(f"✅ Task 1 passed — {len(DOCUMENTS)} documents → {len(chunks)} chunks")
print(f"   Sample chunk: [{chunks[0]['id']}] {chunks[0]['text'][:80]}...")

## Task 2 — Build the Vector Index (10 min)

Store all chunks in a **ChromaDB** collection using SentenceTransformer embeddings.

Steps:
1. Create an embedding function using `SentenceTransformerEmbeddingFunction` with model `"all-MiniLM-L6-v2"`.
2. Create an in-memory ChromaDB client with `chromadb.Client()`.
3. Create a collection named `"bootcamp_kb"` with the embedding function.
4. Add all chunks — you need `ids`, `documents` (the text), and `metadatas` (the doc_id).

> **Hint:** `collection.add(ids=[...], documents=[...], metadatas=[...])`

In [ ]:
def build_vector_index(chunks):
    """
    Embed all chunks and store them in a ChromaDB collection.

    Returns:
        ChromaDB collection object
    """
    # TODO: create the SentenceTransformer embedding function
    ef = ???

    # TODO: create an in-memory ChromaDB client
    client = ???

    # TODO: create a collection named "bootcamp_kb" with the embedding function
    collection = ???

    # TODO: add all chunks to the collection
    collection.add(
        ids       = ???,
        documents = ???,
        metadatas = ???,
    )

    return collection

In [ ]:
# ── TEST 2 ── run this to check your work
collection = build_vector_index(chunks)

count = collection.count()
assert count == len(chunks), f"Expected {len(chunks)} items in the collection, got {count}."

results = collection.query(query_texts=["What is a watermark in streaming?"], n_results=3)
assert len(results["documents"][0]) == 3, "Query should return 3 results."

print(f"✅ Task 2 passed — {count} chunks indexed in ChromaDB")
print(f"   Top result for 'watermark': {results['documents'][0][0][:80]}...")

## Task 3 — BM25 Keyword Search (10 min)

Implement keyword-based search using the **BM25** ranking algorithm.

Steps:
1. Tokenize each chunk's text: lowercase and split by whitespace.
2. Build a `BM25Okapi` index from the tokenized corpus.
3. Get BM25 scores for the query (also tokenized).
4. Return the `top_k` results sorted by score descending.

> **Hint:** `bm25.get_scores(query.lower().split())` returns a score for every chunk.

In [ ]:
def keyword_search(chunks, query, top_k=8):
    """
    Search chunks using BM25 keyword ranking.

    Returns:
        list of (score, chunk_dict) tuples, sorted by score descending
    """
    # TODO: tokenize every chunk (lowercase split)
    tokenized_corpus = ???

    # TODO: build the BM25 index
    bm25 = ???

    # TODO: score the query against all chunks
    scores = ???

    # TODO: pair each score with its chunk, sort descending, return top_k
    ranked = sorted(???, key=lambda x: x[0], reverse=True)
    return ranked[:top_k]

In [ ]:
# ── TEST 3 ── run this to check your work
bm25_results = keyword_search(chunks, "What is a watermark in streaming?", top_k=5)

assert len(bm25_results) == 5, "Should return exactly 5 results."
assert all(isinstance(s, float) and isinstance(c, dict) for s, c in bm25_results), \
    "Each result must be a (float, dict) tuple."
assert bm25_results[0][0] >= bm25_results[-1][0], "Results must be sorted descending by score."

print(f"✅ Task 3 passed — BM25 keyword search working")
print(f"   Top BM25 match (score {bm25_results[0][0]:.3f}): {bm25_results[0][1]['text'][:80]}...")

## Task 4 — Hybrid Search with Reciprocal Rank Fusion (10 min)

Merge the vector results and the BM25 results using **RRF**.

The RRF formula for a document is:

$$\text{score} = \sum_{\text{list}} \frac{1}{k + \text{rank}}$$

Where `k = 60` (standard constant) and `rank` starts at **1** (not 0).

Steps:
1. For each document in the vector hit list, add `1 / (k + rank)` to its running score.
2. Do the same for the BM25 hit list.
3. Sort all documents by their combined RRF score, return top `top_k`.

> **Hint:** Use a dict keyed by chunk id to accumulate scores across both lists.

In [ ]:
def fuse_with_rrf(vector_hits, bm25_hits, k=60, top_k=5):
    """
    Merge vector and BM25 results with Reciprocal Rank Fusion.

    Args:
        vector_hits : list of dicts from collection.query  (keys: 'id', 'document')
        bm25_hits   : list of (score, chunk_dict) tuples from keyword_search

    Returns:
        list of top_k chunk dicts sorted by RRF score descending
    """
    rrf_scores  = {}   # chunk_id → cumulative RRF score
    id_to_chunk = {}   # chunk_id → chunk dict

    # TODO: score vector hits — rank starts at 1
    for rank, hit in enumerate(vector_hits, start=1):
        cid = hit["id"]
        rrf_scores[cid]  = rrf_scores.get(cid, 0.0) + ???
        id_to_chunk[cid] = {"id": cid, "text": hit["document"]}

    # TODO: score BM25 hits — rank starts at 1
    for rank, (_, chunk) in enumerate(bm25_hits, start=1):
        cid = chunk["id"]
        rrf_scores[cid]  = rrf_scores.get(cid, 0.0) + ???
        id_to_chunk[cid] = chunk

    # TODO: sort by RRF score descending and return top_k chunk dicts
    sorted_ids = sorted(rrf_scores, key=lambda x: ???, reverse=True)
    return [id_to_chunk[cid] for cid in sorted_ids[:top_k]]

In [ ]:
# ── TEST 4 ── run this to check your work
query = "How does RRF merge vector and keyword search results?"

vec_raw  = collection.query(query_texts=[query], n_results=8)
vec_hits = [{"id": i, "document": d}
            for i, d in zip(vec_raw["ids"][0], vec_raw["documents"][0])]
bm25_hits = keyword_search(chunks, query, top_k=8)

hybrid = fuse_with_rrf(vec_hits, bm25_hits, top_k=5)

assert len(hybrid) <= 5, "Should return at most 5 results."
assert all("id" in c and "text" in c for c in hybrid), "Each result must have id and text."

print(f"✅ Task 4 passed — RRF fusion working ({len(hybrid)} results)")
print(f"   Top hybrid result: {hybrid[0]['text'][:100]}...")

## Task 5 — Cross-Encoder Reranking (10 min)

Add a **second-stage reranker** that scores each (query, document) pair jointly.

Steps:
1. Create a `CrossEncoder` using model `"cross-encoder/ms-marco-MiniLM-L-6-v2"`.
2. Build a list of `(query, doc_text)` pairs — one per candidate.
3. Call `model.predict(pairs)` to get a relevance score per pair.
4. Sort candidates by score descending and return the top `top_k`.

> **Why?** The cross-encoder sees both the query and document together, giving much
> more accurate relevance scores than cosine similarity between independent embeddings.

In [ ]:
def rerank(query, candidates, top_k=3):
    """
    Rerank candidates using a cross-encoder relevance model.

    Args:
        query      : the user's question (string)
        candidates : list of chunk dicts (output of fuse_with_rrf)
        top_k      : how many to return after reranking

    Returns:
        list of top_k chunk dicts sorted by cross-encoder score descending
    """
    # TODO: load the cross-encoder model
    model = ???

    # TODO: build (query, document_text) pairs for every candidate
    pairs = ???

    # TODO: predict relevance scores for all pairs at once
    scores = ???

    # TODO: zip scores with candidates, sort descending, return top_k dicts
    ranked = sorted(???, key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_k]]

In [ ]:
# ── TEST 5 ── run this to check your work
query      = "What is the difference between tumbling and sliding windows?"
vec_raw    = collection.query(query_texts=[query], n_results=8)
vec_hits   = [{"id": i, "document": d}
               for i, d in zip(vec_raw["ids"][0], vec_raw["documents"][0])]
bm25_hits  = keyword_search(chunks, query, top_k=8)
hybrid     = fuse_with_rrf(vec_hits, bm25_hits, top_k=6)
final_docs = rerank(query, hybrid, top_k=3)

assert len(final_docs) == 3, "Should return exactly 3 results after reranking."
print(f"✅ Task 5 passed — cross-encoder reranking working")
for i, doc in enumerate(final_docs, 1):
    print(f"  [{i}] {doc['text'][:100]}...")

## Task 6 — Evaluation + Full Pipeline (10 min)

**Part A:** Implement `evaluate_precision()` — a simplified version of RAGAS Context Precision.

Context Precision = fraction of retrieved chunks whose cosine similarity with the query exceeds a threshold.

**Part B:** Wire everything together in `run_pipeline()` and run it on three queries.

> **Hint for Part A:** `embed_model.encode(text, normalize_embeddings=True)` returns a unit vector.
> Cosine similarity between two unit vectors = `np.dot(vec_a, vec_b)`.

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")


def evaluate_precision(query, retrieved_docs, threshold=0.30):
    """
    Compute context precision: fraction of retrieved chunks
    with cosine similarity > threshold relative to the query.

    Returns: float between 0.0 and 1.0
    """
    query_emb = embed_model.encode(query, normalize_embeddings=True)

    # TODO: for each doc in retrieved_docs, encode its text and compute cosine similarity
    scores = []
    for doc in retrieved_docs:
        doc_emb = ???
        similarity = ???
        scores.append(similarity)

    # TODO: compute the fraction of scores above the threshold
    precision = ???
    return round(precision, 3)


def run_pipeline(query, top_k_retrieve=8, top_k_rerank=3):
    """
    Full RAG pipeline: query → hybrid retrieval → rerank → prompt → evaluation.
    """
    print(f"\nQUERY: {query}")
    print("-" * 60)

    # Stage 1: Vector search
    vec_raw  = collection.query(query_texts=[query], n_results=top_k_retrieve)
    vec_hits = [{"id": i, "document": d}
                for i, d in zip(vec_raw["ids"][0], vec_raw["documents"][0])]

    # Stage 2: BM25 keyword search
    bm25_hits = keyword_search(chunks, query, top_k=top_k_retrieve)

    # Stage 3: RRF fusion
    hybrid = fuse_with_rrf(vec_hits, bm25_hits, top_k=top_k_retrieve)

    # Stage 4: Cross-encoder rerank
    final_docs = rerank(query, hybrid, top_k=top_k_rerank)

    # Stage 5: Build RAG prompt (in production you would call an LLM here)
    context = "\n\n".join(f"[{i+1}] {d['text']}" for i, d in enumerate(final_docs))
    prompt  = (
        f"Answer based strictly on the context below.\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION: {query}\nANSWER:"
    )

    # Stage 6: Evaluate
    precision = evaluate_precision(query, final_docs)

    print(f"Top {top_k_rerank} chunks after reranking:")
    for i, doc in enumerate(final_docs, 1):
        print(f"  [{i}] {doc['text'][:90]}...")
    print(f"\nContext Precision: {precision}")
    if precision < 0.5:
        print("⚠️  Low precision — try adjusting chunk size or threshold.")
    else:
        print("✅  Good retrieval quality.")
    print(f"\nPrompt (first 300 chars):\n{prompt[:300]}...")
    return final_docs, precision

In [ ]:
# ── TEST 6 ── run the full pipeline on three queries
TEST_QUERIES = [
    "How does Kafka guarantee message ordering?",
    "What is the difference between tumbling and sliding windows?",
    "How does cross-encoder reranking improve RAG accuracy?",
]

all_passed = True
for q in TEST_QUERIES:
    docs, prec = run_pipeline(q)
    if len(docs) != 3 or not (0.0 <= prec <= 1.0):
        all_passed = False
        print(f"❌ Issue with query: {q}")

if all_passed:
    print("\n" + "="*60)
    print("✅ ALL TASKS COMPLETE — Full RAG pipeline working!")
    print("="*60)

## Bonus Challenge (if you finish early)

Pick **one** of the following:

**A — Chunk size experiment**
Run `run_pipeline()` with `chunk_size=1` and `chunk_size=3` and compare the precision scores.
Which produces better results for these queries, and why?

**B — Threshold sensitivity**
Change the `threshold` in `evaluate_precision()` from `0.30` to `0.50`.
How many of your retrieved chunks survive at the higher threshold?
What does this tell you about retrieval quality?

**C — Remove the reranker**
Comment out the rerank step in `run_pipeline()` and pass `hybrid` directly to
`evaluate_precision()`. Does precision go up or down? Explain why.